# Query Classification: A / B / C / U

`baseline.ipynb`의 의료 범위 정의를 그대로 써서, 평가 세트 후보 질문을 LLM(Qwen2.5-7B-Instruct)으로 분류합니다.

**실행 환경**: Mac 로컬. 모델은 Ollama로 띄운 `qwen2.5:7b-instruct`(4-bit 양자화)를 씁니다. 중간 결과를 `raw_<프롬프트버전>.jsonl`에 계속 저장하므로, 중간에 멈춰도 다시 실행하면 이어서 분류합니다. 분류 대상을 늘리면 이미 라벨이 있는 질문은 두고 새 질문만 분류합니다.

| 라벨 | 의미 | 사용처 |
|---|---|---|
| A | 핵심 의료 | 의료 평가 세트 |
| B | 의료 인접 (영양, 운동, 비용, 보험, 수의학 등) | 별도 보관, 검색 성능만 참고 |
| C | 비의료 | 도메인 밖(거절해야 하는) 평가 세트 |
| U | 불확실 | 제외 |

플래그: `high_risk`(개인 의료 판단 요구), `time_sensitive`(바뀔 수 있는 가이드라인)

**분류 대상**: 정답이 있는 질문만 대상으로 합니다.
- `med_url`: 정답 passage가 의료 도메인인 질문. validation 전체와 train 전체 (train은 검색 모델 파인튜닝 데이터로도 씀)
- `no_med`: 의료 도메인 passage가 하나도 없는 질문. validation 전체와 train 1,000개 샘플

분류할 때는 질문과 함께 **데이터셋이 제공한 정답 passage**(없으면 첫 passage)를 보여줍니다. `stroke`처럼 질문만으로는 의미를 정할 수 없는 경우(Case 3)에 판단 근거로 쓰기 위해서입니다.

**검증**: Codex가 같은 기준(`labeling/LABELING_GUIDE.md`)으로 200개를 따로 라벨링하고, 마지막 섹션에서 LLM 라벨과의 일치율을 계산합니다.

**출력**: `RAG-Chat/data/labels/query_labels.parquet` → `LabeledBaseline.ipynb`에서 사용

## 0. 환경 설정

처음 한 번만 터미널에서 실행합니다.
```bash
brew install ollama
brew services start ollama          # 백그라운드 서버 (끌 때: brew services stop ollama)
ollama pull qwen2.5:7b-instruct     # 약 4.7GB
```
병렬 요청으로 속도를 높이려면 서버를 `OLLAMA_NUM_PARALLEL=4 ollama serve`로 띄우세요 (아래 `N_WORKERS`와 맞춤).

In [12]:
%pip install -q -U datasets pyarrow scikit-learn requests tqdm

Note: you may need to restart the kernel to use updated packages.


In [14]:
import requests

OLLAMA_URL = "http://localhost:11434"
try:
    tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    print("Ollama 모델:", [m["name"] for m in tags["models"]])
except requests.ConnectionError:
    raise RuntimeError("Ollama 서버가 꺼져 있습니다. 터미널에서 `brew services start ollama` 실행")

Ollama 모델: ['qwen2.5:7b-instruct']


In [ ]:
import os, re, json, random
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd

# RAG-Chat 폴더 (RAG-Chat 또는 RAG-Chat/Baseline 어디서 열어도 찾음). 다른 곳에 두려면 RAG_BASE_DIR 환경변수 지정
BASE_DIR = Path(os.environ.get("RAG_BASE_DIR", Path.cwd() if (Path.cwd() / "Baseline").exists() else Path.cwd().parent)).resolve()
LABEL_DIR = BASE_DIR / "data" / "labels"
LABEL_DIR.mkdir(parents=True, exist_ok=True)
LABEL_PATH = LABEL_DIR / "query_labels.parquet"
# Codex가 라벨링한 검수 시트
REVIEW_PATH = BASE_DIR / "Baseline" / "labeling" / "review_sample_labeled.csv"

SEED = 42
DATASET_NAME, DATASET_VERSION = "microsoft/ms_marco", "v1.1"
GEN_MODEL = "qwen2.5:7b-instruct"   # Ollama 태그 (Qwen/Qwen2.5-7B-Instruct의 Q4_K_M 양자화)
N_TRAIN_MED = None     # train med_url 샘플 수. None이면 전체 (파인튜닝 학습 데이터용)
N_TRAIN_OOD = 1000     # train no_med 샘플 (도메인 밖 튜닝 세트용)
MAX_NEW_TOKENS = 64
N_WORKERS = 4         # Ollama 동시 요청 수 (OLLAMA_NUM_PARALLEL과 맞춤)

random.seed(SEED)
np.random.seed(SEED)

## 1. 데이터와 후보 질문
코퍼스와 의료 도메인 판정은 `MedicalBaseline.ipynb`와 같은 코드를 씁니다.

In [16]:
MEDICAL_DOMAINS = [
    "webmd.com", "webmd.boots.com", "mayoclinic.org", "mayoclinic.com", "medicinenet.com",
    "healthline.com", "medical-dictionary.thefreedictionary.com", "nih.gov", "medlineplus.gov",
    "cdc.gov", "everydayhealth.com", "nhs.uk", "medicalnewstoday.com", "rxlist.com",
    "medscape.com", "emedicinehealth.com", "wisegeekhealth.com", "health.howstuffworks.com",
    "cancer.org", "clevelandclinic.org", "patient.info", "hopkinsmedicine.org", "healthtap.com",
    "news-medical.net", "cancerresearchuk.org", "americanpregnancy.org", "spine-health.com",
    "who.int", "cancer.gov", "womenshealth.gov", "md-health.com", "heart.org", "healthhype.com",
    "familydoctor.org", "goodrx.com", "medhelp.org", "symptomfind.com", "medilexicon.com",
    "psychcentral.com", "healthcommunities.com", "med-health.net", "diabetes.org",
    "themedicalquestions.com", "health.harvard.edu", "drugs.com", "cancercenter.com",
    "rightdiagnosis.com", "innerbody.com", "labtestsonline.org", "sharecare.com", "umm.edu",
    "kidshealth.org", "health.com", "healthguideinfo.com", "medicalhealthtests.com",
]

def get_host(url):
    host = (urlparse(url).hostname or "").lower()
    return host[4:] if host.startswith("www.") else host

def is_medical_host(host):
    return any(host == d or host.endswith("." + d) for d in MEDICAL_DOMAINS)

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def has_answer(answers):
    return any(
        str(a).strip() and str(a).strip().lower().rstrip(".") != "no answer present"
        for a in (answers if answers is not None else [])
    )

In [17]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, name=DATASET_VERSION)
qa = {split: dataset[split].to_pandas() for split in ("train", "validation")}
for split, df in qa.items():
    print(split, len(df))

train 82326
validation 10047


In [18]:
rows = []
for split, df in qa.items():
    for qid, p in zip(df["query_id"], df["passages"]):
        for text, sel, url in zip(p["passage_text"], p["is_selected"], p["url"]):
            rows.append((split, int(qid), normalize_text(text), int(sel), url))

flat = pd.DataFrame(rows, columns=["split", "query_id", "text", "is_selected", "url"])
flat = flat[flat["text"] != ""]
flat["host"] = flat["url"].map(get_host)
host_is_med = {h: is_medical_host(h) for h in flat["host"].unique()}
flat["is_medical"] = flat["host"].map(host_is_med)

corpus = (
    flat[flat["is_medical"]]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
corpus.insert(0, "pid", np.arange(len(corpus)))   # pid == 행 번호 (인덱스 위치와 동일)
text2pid = dict(zip(corpus["text"], corpus["pid"]))

print(f"전체 passage: {len(flat):,} / 고유 텍스트: {flat['text'].nunique():,}")
print(f"의료 코퍼스: {len(corpus):,}")
print(corpus["host"].value_counts().head(10).to_string())

전체 passage: 758,553 / 고유 텍스트: 690,688
의료 코퍼스: 59,067
host
webmd.com                                   9797
medical-dictionary.thefreedictionary.com    4665
mayoclinic.org                              3972
medicinenet.com                             3766
healthline.com                              2865
ncbi.nlm.nih.gov                            2291
cdc.gov                                     2171
everydayhealth.com                          1858
nhs.uk                                      1405
medicalnewstoday.com                        1350


In [ ]:
def candidate_pool(split):
    df = qa[split]
    fs = flat[flat["split"] == split]
    selected_med = set(fs.loc[fs["is_medical"] & (fs["is_selected"] == 1), "query_id"])
    any_med = set(fs.loc[fs["is_medical"], "query_id"])
    # 분류용 문맥: 정답 passage 우선, 없으면 첫 passage
    context = (fs.sort_values("is_selected", ascending=False, kind="stable")
                 .drop_duplicates("query_id").set_index("query_id")["text"])

    base = pd.DataFrame({
        "query_id": df["query_id"].astype(int),
        "split": split,
        "query": df["query"].map(normalize_text),
        "query_type": df["query_type"],
        "answers": df["answers"].map(list),
    })
    base["pool"] = np.where(base["query_id"].isin(selected_med), "med_url",
                            np.where(base["query_id"].isin(any_med), None, "no_med"))
    base["context"] = base["query_id"].map(context).fillna("")
    return base[df["answers"].map(has_answer) & base["pool"].notna()].reset_index(drop=True)

val_pool, train_pool = candidate_pool("validation"), candidate_pool("train")
candidates = pd.concat([
    val_pool,
    train_pool[train_pool["pool"] == "med_url"].pipe(
        lambda d: d if N_TRAIN_MED is None else d.sample(N_TRAIN_MED, random_state=SEED)),
    train_pool[train_pool["pool"] == "no_med"].sample(N_TRAIN_OOD, random_state=SEED),
], ignore_index=True)
print(candidates.groupby(["split", "pool"]).size())

## 2. LLM 분류

In [20]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

session = requests.Session()

def chat(messages):
    r = session.post(f"{OLLAMA_URL}/api/chat", json={
        "model": GEN_MODEL,
        "messages": messages,
        "stream": False,
        "format": "json",                     # JSON 형식 강제 → 파싱 실패 감소
        "keep_alive": "30m",
        "options": {"temperature": 0, "seed": SEED, "num_predict": MAX_NEW_TOKENS, "num_ctx": 4096},
    }, timeout=300)
    r.raise_for_status()
    return r.json()["message"]["content"].strip()

def generate(messages_list, keys, cache_path):
    """keys별로 결과를 cache_path(jsonl)에 바로 저장. 이미 있는 key는 건너뛰므로 중단 후 재실행하면 이어서 진행."""
    done = {}
    if cache_path.exists():
        with open(cache_path) as f:
            for line in f:
                rec = json.loads(line)
                done[rec["key"]] = rec["raw"]
    todo = [(k, m) for k, m in zip(keys, messages_list) if k not in done]
    print(f"캐시 {len(done):,}개, 새로 분류 {len(todo):,}개")

    with open(cache_path, "a") as f, ThreadPoolExecutor(N_WORKERS) as ex:
        for k, raw in tqdm(zip((k for k, _ in todo), ex.map(chat, (m for _, m in todo))), total=len(todo)):
            done[k] = raw
            f.write(json.dumps({"key": k, "raw": raw}, ensure_ascii=False) + "\n")
            f.flush()
    return [done[k] for k in keys]

print(chat([{"role": "user", "content": 'Return {"ok": true}'}]))   # 모델 로드 확인 (첫 호출은 수십 초)

{"ok": true}


In [21]:
PROMPT_VERSION = "v2"   # v1: A/B/C/U 글자 출력 → 비의료 질문을 B로 몰아서 분류 (Codex 대비 κ 0.57)
LABELS = ("A", "B", "C", "U")
NAME2LABEL = {"core_medical": "A", "health_adjacent": "B", "non_medical": "C", "uncertain": "U"}

SYSTEM_CLS = """You label web search questions for a medical question-answering system.

Decide in this order:
Step 1. Is the question about human health? That means the human body, diseases, symptoms, medicines, medical tests or treatments, nutrition or fitness for health, or healthcare services.
  If not, the label is non_medical. Most search questions are non_medical, for example questions about places, history, people, vehicles, technology, cooking and recipes, what a food is made of, prices of non-medical things, jobs, salaries and education, money and law, language and grammar, plants and animals in nature, or science that is not about human health.
Step 2. If it is about human health, choose:
  core_medical: needs medical knowledge about diseases, symptoms, diagnosis, medical tests, drugs, treatments and procedures, human anatomy and physiology, prevention and infection, or mental disorders.
  health_adjacent: nutrition and diet for general health, exercise and fitness, general wellness habits, food safety, health insurance, cost of medical services, hospital operations, or the health of pets and animals.
Step 3. Use uncertain only if, even after reading the passage, you cannot tell what the question is asking.

Rules:
- Judge the purpose of the question, not its keywords. A question without medical words can be core_medical, and a question with a medical word can be non_medical (for example the history of a medical device, a doctor's biography, or the grammar of a medical term).
- If a word has several meanings, use the passage to decide.
- Whether the passage contains the answer does not matter.

Flags (always false for non_medical and uncertain):
- high_risk: the question asks for a personal medical decision, such as stopping or changing one's own medication or choosing one's own dose.
- time_sensitive: the answer depends on guidelines that may change, such as treatment recommendations, drug dosages or vaccine schedules.

Examples:
"what causes your hands to shake" -> {"about_human_health": true, "label": "core_medical", "high_risk": false, "time_sensitive": false}
"who invented the mri machine" -> {"about_human_health": false, "label": "non_medical", "high_risk": false, "time_sensitive": false}
"how much water should a person drink daily" -> {"about_human_health": true, "label": "health_adjacent", "high_risk": false, "time_sensitive": false}
"should i stop taking my blood pressure medication" -> {"about_human_health": true, "label": "core_medical", "high_risk": true, "time_sensitive": false}
"what is the recommended treatment for hepatitis c" -> {"about_human_health": true, "label": "core_medical", "high_risk": false, "time_sensitive": true}
"how much does an mri cost" -> {"about_human_health": true, "label": "health_adjacent", "high_risk": false, "time_sensitive": false}
"what is the population of ohio" -> {"about_human_health": false, "label": "non_medical", "high_risk": false, "time_sensitive": false}
"how much does a new roof cost" -> {"about_human_health": false, "label": "non_medical", "high_risk": false, "time_sensitive": false}

Output only one JSON object with the keys about_human_health, label, high_risk, time_sensitive."""


def build_cls_messages(query, passage):
    user = f"Question: {query}\nPassage: {passage[:1000]}\n\nReturn the JSON only."
    return [{"role": "system", "content": SYSTEM_CLS}, {"role": "user", "content": user}]


def _as_bool(v):
    return v is True or str(v).strip().lower() == "true"


def parse_label(text):
    """LLM 출력 → A/B/C/U와 플래그. 라벨 이름과 글자 둘 다 받고, 파싱에 실패하면 U."""
    m = re.search(r"\{.*?\}", text, flags=re.S)
    try:
        d = json.loads(m.group(0))
        raw = str(d.get("label", "")).strip()
        label = NAME2LABEL.get(raw.lower(), raw.upper())
        if label in LABELS:
            flags_ok = label in ("A", "B")
            return {"label": label,
                    "high_risk": flags_ok and _as_bool(d.get("high_risk")),
                    "time_sensitive": flags_ok and _as_bool(d.get("time_sensitive")),
                    "parse_ok": True}
    except (AttributeError, ValueError):
        pass
    for name, label in NAME2LABEL.items():
        if name in text:
            return {"label": label, "high_risk": False, "time_sensitive": False, "parse_ok": False}
    return {"label": "U", "high_risk": False, "time_sensitive": False, "parse_ok": False}

def classify(queries, contexts, keys):
    cache_path = LABEL_DIR / f"raw_{PROMPT_VERSION}.jsonl"
    raw = generate([build_cls_messages(q, c) for q, c in zip(queries, contexts)], keys, cache_path)
    out = pd.DataFrame([parse_label(r) for r in raw])
    out["raw"] = raw
    return out

In [22]:
print(build_cls_messages("what is a stroke", "A stroke occurs when blood flow to the brain is blocked.")[1]["content"])

Question: what is a stroke
Passage: A stroke occurs when blood flow to the brain is blocked.

Return the JSON only.


In [ ]:
labels = None
if LABEL_PATH.exists():
    labels = pd.read_parquet(LABEL_PATH)
    old_version = labels["prompt_version"].iloc[0] if "prompt_version" in labels else "v1"
    if old_version != PROMPT_VERSION:   # 이전 프롬프트 결과는 백업하고 새로 분류
        backup = LABEL_PATH.with_name(f"query_labels_{old_version}.parquet")
        LABEL_PATH.rename(backup)
        print(f"{old_version} 라벨을 {backup.name}로 백업하고 {PROMPT_VERSION}로 다시 분류합니다")
        labels = None

# 이미 라벨이 있는 질문은 그대로 두고, 분류 대상에 새로 들어온 질문만 분류
def label_key(df):
    return df["split"] + ":" + df["query_id"].astype(str)

done = set(label_key(labels)) if labels is not None else set()
new = candidates[~label_key(candidates).isin(done)].reset_index(drop=True)
print(f"기존 라벨 {len(done):,}개, 새로 분류 {len(new):,}개")
if len(new):
    pred = classify(new["query"].tolist(), new["context"].tolist(), label_key(new).tolist())
    new_labels = pd.concat([new, pred], axis=1).assign(prompt_version=PROMPT_VERSION, gen_model=GEN_MODEL)
    labels = pd.concat([labels, new_labels], ignore_index=True) if labels is not None else new_labels
    labels.to_parquet(LABEL_PATH, index=False)
print("전체 라벨:", len(labels), "/ JSON 파싱 실패:", (~labels["parse_ok"]).sum())

## 3. 분류 결과 분포

In [24]:
display(pd.crosstab([labels["split"], labels["pool"]], labels["label"], margins=True))
display(pd.crosstab(labels["pool"], labels["label"], normalize="index").round(3))
print("high_risk:", labels["high_risk"].sum(), "/ time_sensitive:", labels["time_sensitive"].sum())

pd.set_option("display.max_colwidth", 120)
for (pool, lab), g in labels.groupby(["pool", "label"]):
    print(f"\n=== {pool} / {lab} ({len(g)}) ===")
    print("\n".join(g.sample(min(8, len(g)), random_state=SEED)["query"]))

label                  A    B     C    All
split      pool                           
train      med_url  1133  229   138   1500
           no_med     18   37   945   1000
validation med_url   598  143    99    840
           no_med    210  325  6967   7502
All                 1959  734  8149  10842

label,A,B,C
pool,,,
med_url,0.740,0.159,0.101
no_med,0.027,0.043,0.931


high_risk: 3 / time_sensitive: 15

=== med_url / A (1731) ===
how long is the waiting list for a liver
what is the main function of the periosteum
what year was the mmr vaccine introduced
which side is appendix pain
what is the symptoms of hemianopsia
what is a endoscopic retrograde cholangiopancreatography
what is nitrofurantoin used for
definition of mrsa disease

=== med_url / B (372) ===
what is fibre for
types of lymphatic tissue
what is belching
what foods contain monounsaturated fats
are eggs a good diet food
how much exercise do i need
is menstruation the normal condition for humans
is chlorophyll bad for humans

=== med_url / C (237) ===
average weight of newborn kitten
what food has vitamin e
define allergen
what oils contain vitamin e
Quizlet define gene mutation
what is a vagus nerve
what is the name of the sugar molecule in the dna helix
what is the meaning of opium

=== no_med / A (228) ===
what digestive organ produces intrinsic factor
food that contains toxins
how to ge

## 4. Codex 라벨과 비교 (검증)

1. Codex가 `Baseline/labeling/review_sample.csv`를 가이드대로 채워 `review_sample_labeled.csv`로 저장합니다.
2. 아래 셀에서 정확도, Cohen's κ, 혼동행렬, 불일치 사례를 확인합니다.

검수 시트는 validation 후보에서 뽑았으므로(`med_url` 120개, `no_med` 80개) 모두 위 분류 대상에 들어 있습니다.

In [25]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report, confusion_matrix

if not REVIEW_PATH.exists():
    print("검수 파일이 아직 없습니다:", REVIEW_PATH)
else:
    review = pd.read_csv(REVIEW_PATH)
    review["label"] = review["label"].astype(str).str.strip().str.upper()
    merged = review.merge(labels[["query_id", "pool", "label", "high_risk", "time_sensitive"]],
                          on="query_id", suffixes=("_codex", "_llm"))
    print(f"매칭: {len(merged)} / {len(review)}")
    y_true, y_pred = merged["label_codex"], merged["label_llm"]
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}   Cohen's kappa: {cohen_kappa_score(y_true, y_pred):.3f}")
    print(classification_report(y_true, y_pred, labels=list(LABELS), zero_division=0))
    display(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=list(LABELS)),
                         index=[f"codex_{l}" for l in LABELS], columns=[f"llm_{l}" for l in LABELS]))

    # 실제로 쓰는 구분(A vs C)에서의 일치도: 의료 평가 세트와 도메인 밖 세트가 얼마나 정확한지
    ac = merged[y_true.isin(["A", "C"]) | y_pred.isin(["A", "C"])]
    print(f"A/C 관련 {len(ac)}건 중 일치 {(ac['label_codex'] == ac['label_llm']).mean():.3f}")

    disagree = merged[y_true != y_pred][["query", "pool", "label_codex", "label_llm", "note"]]
    disagree.to_csv(LABEL_DIR / "review_disagreements.csv", index=False, encoding="utf-8-sig")
    display(disagree)

검수 파일이 아직 없습니다: /Users/lsm/Desktop/WorkSpace/NLP/Baseline/labeling/review_sample_labeled.csv
